# 03 — Feature Engineering and SageMaker Feature Store

This notebook implements the Week module deliverable:

> Perform feature engineering on raw data and store it in a Feature Store. Split your feature data into training (~40%), test (~10%), validation (~10%) datasets. Reserve some data for "production data" (~40%).

Steps:

1. Pull the cleaned Parquet review data from Athena into a pandas DataFrame.
2. Build sentiment labels from star ratings and exclude 3-star neutral reviews.
3. Balance the two classes (≥10,000 records per class).
4. Engineer numeric and categorical features.
5. Assign a `split_type` of `train`, `validation`, `test`, or `production` to each record using a 40/10/10/40 split.
6. Create a SageMaker Feature Group and ingest the engineered records (offline store on S3).
7. Sanity-check the feature group with `describe()` and a sample Athena query against the offline store.

In [ ]:
%pip install --disable-pip-version-check --quiet "sagemaker>=2.230.0,<3.0.0" awswrangler PyAthena scikit-learn
print("Install complete. IMPORTANT: restart the kernel now before running the next cell.")

In [ ]:
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from time import gmtime, strftime

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sagemaker
import seaborn as sns
from pyathena import connect
from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum
from sagemaker.feature_store.feature_group import FeatureGroup

from plot_style import apply_style, PURPLE_SENTIMENT, PURPLE_SPLITS

%store -r bucket
%store -r region
%store -r database_name
%store -r s3_staging_dir
%store -r parquet_table_name

apply_style()
report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)
table = f"{database_name}.{parquet_table_name}"
print("Source table:", table)
print("Bucket:      ", bucket)
print("Role:        ", role)

## 1. Pull a labeled, balanced sample

We pull enough rows to comfortably exceed the rubric requirement of ≥10,000 records per class even after the 40/10/10/40 split.

In [ ]:
ROWS_PER_CLASS = 25000

positive_df = pd.read_sql(
    f"SELECT review_id, business_id, user_id, stars, review_text, date, review_char_length "
    f"FROM {table} WHERE stars >= 4 LIMIT {ROWS_PER_CLASS}",
    conn,
)
negative_df = pd.read_sql(
    f"SELECT review_id, business_id, user_id, stars, review_text, date, review_char_length "
    f"FROM {table} WHERE stars <= 2 LIMIT {ROWS_PER_CLASS}",
    conn,
)

print(f"Positive rows: {len(positive_df):,}")
print(f"Negative rows: {len(negative_df):,}")

raw_df = pd.concat([positive_df, negative_df], ignore_index=True)
raw_df = raw_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
raw_df.head()

## 2. Feature engineering

- `sentiment_label` (int): 1 for positive, 0 for negative — used as the model target.
- `sentiment` (string): "positive" / "negative" for human-readable analysis.
- `clean_text` (string): lowercased, whitespace-normalized review text.
- `review_word_count` (int): number of word tokens in the cleaned text.
- `review_char_length` (int): already provided by Athena.
- `has_exclamation` / `has_question` (int): simple punctuation signals that correlate with sentiment.
- `review_year` (int): year extracted from the review date, useful for drift monitoring later.

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower().strip()
    return re.sub(r"\s+", " ", text)

def word_count(text: str) -> int:
    return len(re.findall(r"[a-z']+", text or ""))

features_df = raw_df.copy()
features_df["sentiment"] = np.where(features_df["stars"] >= 4, "positive", "negative")
features_df["sentiment_label"] = (features_df["sentiment"] == "positive").astype(int)
features_df["clean_text"] = features_df["review_text"].astype(str).map(clean_text)
features_df["review_word_count"] = features_df["clean_text"].map(word_count)
features_df["has_exclamation"] = features_df["clean_text"].str.contains("!", regex=False).astype(int)
features_df["has_question"] = features_df["clean_text"].str.contains("?", regex=False).astype(int)
features_df["review_year"] = pd.to_datetime(features_df["date"], errors="coerce").dt.year.fillna(0).astype(int)

features_df = features_df.dropna(subset=["review_id", "clean_text"])
features_df = features_df[features_df["clean_text"].str.len() > 0].reset_index(drop=True)

print("Engineered feature counts by sentiment:")
print(features_df["sentiment"].value_counts())
features_df.head()

## 3. Assign 40 / 10 / 10 / 40 split

We stratify by `sentiment_label` so each split keeps the same class balance.

In [ ]:
from sklearn.model_selection import train_test_split

SPLIT_SHARES = {"train": 0.40, "validation": 0.10, "test": 0.10, "production": 0.40}
RANDOM_STATE = 42

train_idx, remainder_idx = train_test_split(
    features_df.index,
    train_size=SPLIT_SHARES["train"],
    stratify=features_df["sentiment_label"],
    random_state=RANDOM_STATE,
)
remainder = features_df.loc[remainder_idx]

production_share_of_remainder = SPLIT_SHARES["production"] / (1 - SPLIT_SHARES["train"])
production_idx, val_test_idx = train_test_split(
    remainder.index,
    train_size=production_share_of_remainder,
    stratify=remainder["sentiment_label"],
    random_state=RANDOM_STATE,
)

val_test = features_df.loc[val_test_idx]
val_idx, test_idx = train_test_split(
    val_test.index,
    train_size=0.5,
    stratify=val_test["sentiment_label"],
    random_state=RANDOM_STATE,
)

features_df.loc[train_idx, "split_type"] = "train"
features_df.loc[val_idx, "split_type"] = "validation"
features_df.loc[test_idx, "split_type"] = "test"
features_df.loc[production_idx, "split_type"] = "production"

split_summary = features_df.groupby("split_type").agg(
    row_count=("review_id", "count"),
    positive_share=("sentiment_label", "mean"),
)
split_summary["share_of_total"] = (split_summary["row_count"] / len(features_df)).round(4)
split_summary

### Visualize the 40 / 10 / 10 / 40 split

Saved to `reports/feature_split_distribution.png` for the ML Design Document.

In [ ]:
split_order = ["train", "validation", "test", "production"]
plot_df = split_summary.reindex(split_order).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=plot_df,
    x="split_type",
    y="row_count",
    hue="split_type",
    palette=PURPLE_SPLITS,
    order=split_order,
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Rows per Split")
axes[0].set_xlabel("Split")
axes[0].set_ylabel("Row count")
for i, row in plot_df.iterrows():
    axes[0].text(i, row["row_count"], f"{int(row['row_count']):,}", ha="center", va="bottom")

sns.barplot(
    data=plot_df,
    x="split_type",
    y="positive_share",
    hue="split_type",
    palette=PURPLE_SPLITS,
    order=split_order,
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Positive Class Share per Split")
axes[1].set_xlabel("Split")
axes[1].set_ylabel("Positive share")
axes[1].set_ylim(0, 1)
for i, row in plot_df.iterrows():
    axes[1].text(i, row["positive_share"], f"{row['positive_share']:.1%}", ha="center", va="bottom")

plt.tight_layout()
plt.savefig(report_dir / "feature_split_distribution.png", dpi=150)
plt.show()

## 4. Add Feature Store required columns

Every Feature Group needs a record identifier and an event time. We use `review_id` as the identifier and the current UTC timestamp as the event time.

In [ ]:
features_df["event_time"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

FEATURE_COLUMNS = [
    "review_id", "business_id", "user_id", "stars", "sentiment", "sentiment_label",
    "review_text", "clean_text", "review_char_length", "review_word_count",
    "has_exclamation", "has_question", "review_year", "split_type", "event_time",
]
features_df = features_df[FEATURE_COLUMNS]
features_df.dtypes

## 5. Create the SageMaker Feature Group

In [ ]:
timestamp = strftime("%d-%H-%M-%S", gmtime())
feature_group_name = f"yelp-sentiment-feature-group-{timestamp}"

feature_definitions = [
    FeatureDefinition("review_id", FeatureTypeEnum.STRING),
    FeatureDefinition("business_id", FeatureTypeEnum.STRING),
    FeatureDefinition("user_id", FeatureTypeEnum.STRING),
    FeatureDefinition("stars", FeatureTypeEnum.INTEGRAL),
    FeatureDefinition("sentiment", FeatureTypeEnum.STRING),
    FeatureDefinition("sentiment_label", FeatureTypeEnum.INTEGRAL),
    FeatureDefinition("review_text", FeatureTypeEnum.STRING),
    FeatureDefinition("clean_text", FeatureTypeEnum.STRING),
    FeatureDefinition("review_char_length", FeatureTypeEnum.INTEGRAL),
    FeatureDefinition("review_word_count", FeatureTypeEnum.INTEGRAL),
    FeatureDefinition("has_exclamation", FeatureTypeEnum.INTEGRAL),
    FeatureDefinition("has_question", FeatureTypeEnum.INTEGRAL),
    FeatureDefinition("review_year", FeatureTypeEnum.INTEGRAL),
    FeatureDefinition("split_type", FeatureTypeEnum.STRING),
    FeatureDefinition("event_time", FeatureTypeEnum.STRING),
]

feature_group = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=sagemaker_session,
    feature_definitions=feature_definitions,
)

feature_group.create(
    record_identifier_name="review_id",
    event_time_feature_name="event_time",
    role_arn=role,
    s3_uri=f"s3://{bucket}/feature-store/yelp-sentiment/{feature_group_name}",
    enable_online_store=False,
)

while feature_group.describe()["FeatureGroupStatus"] == "Creating":
    print("Waiting for feature group creation...")
    time.sleep(5)

print("Feature group ready:", feature_group_name)
%store feature_group_name

## 6. Ingest the features

The Feature Store needs all column dtypes to match the feature definitions exactly. We cast a few columns explicitly to make this deterministic.

In [ ]:
ingest_df = features_df.copy()
for col in ["stars", "sentiment_label", "review_char_length", "review_word_count", "has_exclamation", "has_question", "review_year"]:
    ingest_df[col] = ingest_df[col].astype("int64")
for col in ["review_id", "business_id", "user_id", "sentiment", "review_text", "clean_text", "split_type", "event_time"]:
    ingest_df[col] = ingest_df[col].astype(str)

ingest_df = ingest_df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
print(f"Ingesting {len(ingest_df):,} records...")

ingestion_manager = feature_group.ingest(data_frame=ingest_df, max_workers=4, wait=True)
print("Ingestion finished. Failed indices:", getattr(ingestion_manager, "_failed_indices", "unknown"))

## 7. Describe and verify

In [ ]:
description = feature_group.describe()
{k: description[k] for k in ["FeatureGroupName", "RecordIdentifierFeatureName", "EventTimeFeatureName", "FeatureGroupStatus", "OfflineStoreConfig"]}

In [ ]:
offline_table = description["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]
offline_db = description["OfflineStoreConfig"]["DataCatalogConfig"]["Database"]
print(f"Offline store table: {offline_db}.{offline_table}")
print("Note: it can take a few minutes for offline data to land in S3 before this Athena query returns rows.")
%store offline_table
%store offline_db

In [ ]:
split_dist = features_df["split_type"].value_counts().rename_axis("split_type").reset_index(name="row_count")
split_dist["share"] = (split_dist["row_count"] / split_dist["row_count"].sum()).round(4)
split_dist

## Persist split CSVs to S3 for downstream notebooks

Materializing each split to its own S3 prefix means later notebooks (training, monitoring, batch inference) can read each split with a single `pd.read_csv` without needing to query the Feature Store every time.

In [ ]:
import awswrangler as wr

split_prefix = "processed/splits"
for split_name in ["train", "validation", "test", "production"]:
    subset = features_df[features_df["split_type"] == split_name]
    target = f"s3://{bucket}/{split_prefix}/{split_name}/{split_name}.csv"
    wr.s3.to_csv(df=subset, path=target, index=False)
    print(f"Wrote {len(subset):,} rows to {target}")
%store split_prefix

## Done

Continue to `04_dataset_splits.ipynb` to verify the splits and produce the final summary table for the design document.